# HonestDiD Python Package - Validation Against R

This notebook verifies that the Python implementation of HonestDiD produces results matching the original R package.

## Instructions

1. First, run the R script `test_honestdid.R` to generate the test data and R results
2. Then run this notebook to compare Python results against R

```bash
cd tests
Rscript test_honestdid.R
```

In [4]:
import os
import json
import numpy as np
import pandas as pd
import torch
import sys

# Suppress duplicate library warning
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
sys.path.insert(0, '/Users/anzony.quisperojas/Documents/GitHub/python/honestdid')

import honestdid as hd

print(f"HonestDiD Python version: {hd.__version__}")

HonestDiD Python version: 0.1.0


## Load Test Data from R

Load the BCdata_EventStudy exported from R.

In [ ]:
# Load BCdata from R export
with open('../_data/test_data_bcdata.json', 'r') as f:
    bc_data = json.load(f)

betahat = torch.tensor(bc_data['betahat'], dtype=torch.float64)
sigma = torch.tensor(bc_data['sigma'], dtype=torch.float64)

print(f"betahat length: {len(betahat)}")
print(f"sigma dimensions: {sigma.shape}")
print(f"\nbetahat values:")
print(betahat.numpy().round(6))

betahat length: 8
sigma dimensions: torch.Size([8, 8])

betahat values:
[ 0.006696  0.029345 -0.006473  0.073015  0.195961  0.312064  0.239542
  0.126043]


In [8]:
# Load R results for comparison
with open('../_data/r_test_results.json', 'r') as f:
    r_results = json.load(f)

print("R results loaded successfully")
print("Available tests:", list(r_results['bcdata'].keys()))

R results loaded successfully
Available tests: ['originalCS', 'deltaSD', 'deltaRM', 'flci', 'M_upperBound']


In [6]:
# Set parameters (same as R)
numPrePeriods = 4
numPostPeriods = 4
l_vec = hd.basis_vector(1, numPostPeriods)  # First post-period effect
alpha = 0.05

---
## Test 1: Original Confidence Set

Compare the original CS (assuming parallel trends holds exactly).

In [11]:
# Python result
originalCS_py = hd.constructOriginalCS(
    betahat=betahat,
    sigma=sigma,
    numPrePeriods=numPrePeriods,
    numPostPeriods=numPostPeriods,
    l_vec=l_vec,
    alpha=alpha
)

py_lb = float(originalCS_py['lb'].iloc[0])
py_ub = float(originalCS_py['ub'].iloc[0])

# R result
r_lb = r_results['bcdata']['originalCS']['lb'][0][0]
r_ub = r_results['bcdata']['originalCS']['ub'][0][0]

print("=" * 60)
print("Test 1: Original Confidence Set")
print("=" * 60)
print(f"\nPython: [{py_lb:.6f}, {py_ub:.6f}]")
print(f"R:      [{r_lb:.6f}, {r_ub:.6f}]")
print(f"\nDifference (lb): {abs(py_lb - r_lb):.2e}")
print(f"Difference (ub): {abs(py_ub - r_ub):.2e}")

# Check if close
tol = 1e-4
match = abs(py_lb - r_lb) < tol and abs(py_ub - r_ub) < tol
print(f"\n{'✓ PASS' if match else '✗ FAIL'}: Results match within tolerance {tol}")

Test 1: Original Confidence Set

Python: [0.158775, 0.233147]
R:      [0.158775, 0.233147]

Difference (lb): 2.78e-17
Difference (ub): 2.78e-17

✓ PASS: Results match within tolerance 0.0001


---
## Test 2: Sensitivity Results - DeltaSD (Smoothness)

Compare sensitivity analysis under smoothness restrictions.

In [12]:
# Python result
Mvec = [0, 0.01, 0.02, 0.03]
delta_sd_py = hd.createSensitivityResults(
    betahat=betahat,
    sigma=sigma,
    numPrePeriods=numPrePeriods,
    numPostPeriods=numPostPeriods,
    l_vec=l_vec,
    Mvec=Mvec,
    alpha=alpha
)

print("=" * 60)
print("Test 2: Sensitivity Results (DeltaSD - Smoothness)")
print("=" * 60)
print("\nPython Results:")
print(delta_sd_py.to_string(index=False))

/opt/anaconda3/envs/package-testing/lib/python3.8/site-packages/cvxpy/reductions/solvers/solving_chain.py:356: FutureWarning: 
    You specified your problem should be solved by ECOS. Starting in
    CXVPY 1.6.0, ECOS will no longer be installed by default with CVXPY.
    Please either add ECOS as an explicit install dependency to your project
    or switch to our new default solver, Clarabel, by either not specifying a
    solver argument or specifying ``solver=cp.CLARABEL``. To suppress this
    warning while continuing to use ECOS, you can filter this warning using
    Python's ``warnings`` module until you are using 1.6.0.
    
  warnings.warn(ECOS_DEP_DEPRECATION_MSG, FutureWarning)


Test 2: Sensitivity Results (DeltaSD - Smoothness)

Python Results:
      lb       ub method   Delta    M
0.131538 0.216109   FLCI DeltaSD 0.00
0.154596 0.270168   FLCI DeltaSD 0.01
0.176557 0.315361   FLCI DeltaSD 0.02
0.189340 0.348612   FLCI DeltaSD 0.03


In [13]:
# R result
r_deltaSD = pd.DataFrame(r_results['bcdata']['deltaSD'])
print("\nR Results:")
print(r_deltaSD.to_string(index=False))


R Results:
      lb       ub method   Delta    M
0.131411 0.216170   FLCI DeltaSD 0.00
0.154965 0.270385   FLCI DeltaSD 0.01
0.177301 0.315956   FLCI DeltaSD 0.02
0.189404 0.348548   FLCI DeltaSD 0.03


In [14]:
# Compare
print("\nComparison (Python - R):")
comparison_sd = pd.DataFrame({
    'M': Mvec,
    'lb_diff': delta_sd_py['lb'].values - r_deltaSD['lb'].values,
    'ub_diff': delta_sd_py['ub'].values - r_deltaSD['ub'].values
})
print(comparison_sd.to_string(index=False))

max_diff = max(comparison_sd['lb_diff'].abs().max(), comparison_sd['ub_diff'].abs().max())
print(f"\nMax absolute difference: {max_diff:.2e}")
print(f"{'✓ PASS' if max_diff < tol else '✗ FAIL'}: Results match within tolerance {tol}")


Comparison (Python - R):
   M   lb_diff   ub_diff
0.00  0.000127 -0.000061
0.01 -0.000369 -0.000217
0.02 -0.000744 -0.000596
0.03 -0.000065  0.000065

Max absolute difference: 7.44e-04
✗ FAIL: Results match within tolerance 0.0001


---
## Test 3: Sensitivity Results - DeltaRM (Relative Magnitudes)

Compare sensitivity analysis under relative magnitudes restrictions.

In [15]:
# Python result
Mbarvec = [0.5, 1.0, 1.5, 2.0]
delta_rm_py = hd.createSensitivityResults_relativeMagnitudes(
    betahat=betahat,
    sigma=sigma,
    numPrePeriods=numPrePeriods,
    numPostPeriods=numPostPeriods,
    l_vec=l_vec,
    Mbarvec=Mbarvec,
    alpha=alpha
)

print("=" * 60)
print("Test 3: Sensitivity Results (DeltaRM - Relative Magnitudes)")
print("=" * 60)
print("\nPython Results:")
print(delta_rm_py.to_string(index=False))

/Users/anzony.quisperojas/Documents/GitHub/python/honestdid/honestdid/core.py:1108: UserWarning: CI is open at one of the endpoints; CI length may not be accurate
  warnings.warn("CI is open at one of the endpoints; CI length may not be accurate")


Test 3: Sensitivity Results (DeltaRM - Relative Magnitudes)

Python Results:
       lb       ub method   Delta  Mbar
 0.118889 0.271584   C-LF DeltaRM   0.5
 0.067231 0.318684   C-LF DeltaRM   1.0
 0.013294 0.369582   C-LF DeltaRM   1.5
-0.041402 0.379458   C-LF DeltaRM   2.0


In [16]:
# R result
r_deltaRM = pd.DataFrame(r_results['bcdata']['deltaRM'])
print("\nR Results:")
print(r_deltaRM.to_string(index=False))


R Results:
       lb       ub method   Delta  Mbar
 0.118129 0.271584   C-LF DeltaRM   0.5
 0.067231 0.318684   C-LF DeltaRM   1.0
 0.013294 0.369582   C-LF DeltaRM   1.5
-0.042162 0.379458   C-LF DeltaRM   2.0


In [17]:
# Compare
print("\nComparison (Python - R):")
comparison_rm = pd.DataFrame({
    'Mbar': Mbarvec,
    'lb_diff': delta_rm_py['lb'].values - r_deltaRM['lb'].values,
    'ub_diff': delta_rm_py['ub'].values - r_deltaRM['ub'].values
})
print(comparison_rm.to_string(index=False))

max_diff = max(comparison_rm['lb_diff'].abs().max(), comparison_rm['ub_diff'].abs().max())
print(f"\nMax absolute difference: {max_diff:.2e}")
print(f"{'✓ PASS' if max_diff < tol else '✗ FAIL'}: Results match within tolerance {tol}")


Comparison (Python - R):
 Mbar  lb_diff  ub_diff
  0.5  0.00076      0.0
  1.0  0.00000      0.0
  1.5  0.00000      0.0
  2.0  0.00076      0.0

Max absolute difference: 7.60e-04
✗ FAIL: Results match within tolerance 0.0001


---
## Test 4: FLCI (Fixed-Length Confidence Interval)

Compare the optimal fixed-length confidence interval.

In [38]:
# Python result
flci_py = hd.find_optimal_flci(
    betahat=betahat,
    sigma=sigma,
    numPrePeriods=numPrePeriods,
    numPostPeriods=numPostPeriods,
    l_vec=l_vec,
    M=0,
    alpha=alpha
)


/opt/anaconda3/envs/package-testing/lib/python3.8/site-packages/cvxpy/reductions/solvers/solving_chain.py:356: FutureWarning: 
    You specified your problem should be solved by ECOS. Starting in
    CXVPY 1.6.0, ECOS will no longer be installed by default with CVXPY.
    Please either add ECOS as an explicit install dependency to your project
    or switch to our new default solver, Clarabel, by either not specifying a
    solver argument or specifying ``solver=cp.CLARABEL``. To suppress this
    warning while continuing to use ECOS, you can filter this warning using
    Python's ``warnings`` module until you are using 1.6.0.
    
  warnings.warn(ECOS_DEP_DEPRECATION_MSG, FutureWarning)


In [22]:

# R result
r_flci = r_results['bcdata']['flci']

print("=" * 60)
print("Test 4: FLCI (Fixed-Length Confidence Interval)")
print("=" * 60)
print(f"\nPython:")
print(f"  Optimal Half-Length: {flci_py['optimalHalfLength']:.6f}")
print(f"  FLCI: [{flci_py['FLCI'][0]:.6f}, {flci_py['FLCI'][1]:.6f}]")
print(f"\nR:")
print(f"  Optimal Half-Length: {r_flci['optimalHalfLength'][0]:.6f}")
print(f"  FLCI: [{r_flci['FLCI'][0]:.6f}, {r_flci['FLCI'][1]:.6f}]")

hl_diff = abs(flci_py['optimalHalfLength'] - r_flci['optimalHalfLength'][0])
flci_lb_diff = abs(flci_py['FLCI'][0] - r_flci['FLCI'][0])
flci_ub_diff = abs(flci_py['FLCI'][1] - r_flci['FLCI'][1])

print(f"\nDifferences:")
print(f"  Half-Length: {hl_diff:.2e}")
print(f"  FLCI lb: {flci_lb_diff:.2e}")
print(f"  FLCI ub: {flci_ub_diff:.2e}")

max_diff = max(hl_diff, flci_lb_diff, flci_ub_diff)
print(f"\n{'✓ PASS' if max_diff < tol else '✗ FAIL'}: Results match within tolerance {tol}")

Test 4: FLCI (Fixed-Length Confidence Interval)

Python:
  Optimal Half-Length: 0.042285
  FLCI: [0.131538, 0.216109]

R:
  Optimal Half-Length: 0.042379
  FLCI: [0.131411, 0.216170]

Differences:
  Half-Length: 9.37e-05
  FLCI lb: 1.27e-04
  FLCI ub: 6.09e-05

✗ FAIL: Results match within tolerance 0.0001


---
## Test 5: DeltaSD Upper Bound for M

Compare the upper bound for M from pre-treatment periods.

In [24]:
# Python result
M_ub_py = hd.DeltaSD_upperBound_Mpre(
    betahat=betahat,
    sigma=sigma,
    numPrePeriods=numPrePeriods,
    alpha=alpha
)

# R result
M_ub_r = r_results['bcdata']['M_upperBound'][0]

print("=" * 60)
print("Test 5: DeltaSD Upper Bound for M")
print("=" * 60)
print(f"\nPython: {M_ub_py:.6f}")
print(f"R:      {M_ub_r:.6f}")
print(f"\nDifference: {abs(M_ub_py - M_ub_r):.2e}")

match = abs(M_ub_py - M_ub_r) < tol
print(f"\n{'✓ PASS' if match else '✗ FAIL'}: Results match within tolerance {tol}")

Test 5: DeltaSD Upper Bound for M

Python: 0.202170
R:      0.202170

Difference: 2.78e-17

✓ PASS: Results match within tolerance 0.0001


---
## Test 6: Conditional CS with DeltaSD (M=0)

Compare the conditional confidence set.

In [25]:
# Python result
cs_sd_py = hd.computeConditionalCS_DeltaSD(
    betahat=betahat,
    sigma=sigma,
    numPrePeriods=numPrePeriods,
    numPostPeriods=numPostPeriods,
    l_vec=l_vec,
    alpha=alpha,
    M=0,
    hybrid_flag="FLCI"
)

accepted = cs_sd_py[cs_sd_py['accept'] == 1]['grid']

print("=" * 60)
print("Test 6: Conditional CS - DeltaSD (M=0)")
print("=" * 60)
if len(accepted) > 0:
    print(f"\nPython CS: [{accepted.min():.6f}, {accepted.max():.6f}]")
else:
    print("\nPython CS: Empty set")

print("\n(Compare with R output from running the R script)")

Test 6: Conditional CS - DeltaSD (M=0)

Python CS: [0.205952, 0.234377]

(Compare with R output from running the R script)


---
## Summary

Run all tests and summarize results.

In [26]:
print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)

tests = [
    ("Original CS", abs(py_lb - r_lb) < tol and abs(py_ub - r_ub) < tol),
    ("DeltaSD Sensitivity", comparison_sd['lb_diff'].abs().max() < tol and comparison_sd['ub_diff'].abs().max() < tol),
    ("DeltaRM Sensitivity", comparison_rm['lb_diff'].abs().max() < tol and comparison_rm['ub_diff'].abs().max() < tol),
    ("FLCI", max(hl_diff, flci_lb_diff, flci_ub_diff) < tol),
    ("M Upper Bound", abs(M_ub_py - M_ub_r) < tol),
]

print(f"\nTolerance: {tol}")
print("\nResults:")
all_pass = True
for name, passed in tests:
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {status}: {name}")
    all_pass = all_pass and passed

print("\n" + "=" * 60)
if all_pass:
    print("ALL TESTS PASSED - Python matches R implementation!")
else:
    print("SOME TESTS FAILED - Check differences above")
print("=" * 60)

VALIDATION SUMMARY

Tolerance: 0.0001

Results:
  ✓ PASS: Original CS
  ✗ FAIL: DeltaSD Sensitivity
  ✗ FAIL: DeltaRM Sensitivity
  ✗ FAIL: FLCI
  ✓ PASS: M Upper Bound

SOME TESTS FAILED - Check differences above


---
## Part 2: Medicaid Expansion Example (Optional)

If you ran the R script with the Medicaid data, you can compare those results too.

In [33]:
# Try to load Medicaid data
try:
    with open('../_data/test_data_medicaid.json', 'r') as f:
        med_data = json.load(f)
    
    betahat_med = torch.tensor(med_data['betahat'], dtype=torch.float64)
    sigma_med = torch.tensor(med_data['sigma'], dtype=torch.float64)
    
    print("Medicaid data loaded successfully")
    print(f"betahat length: {len(betahat_med)}")
    print(f"sigma dimensions: {sigma_med.shape}")
    print(f"\nbetahat values:")
    print(betahat_med.numpy().round(6))
    
    medicaid_available = True
except FileNotFoundError:
    print("Medicaid data not available. Run R script first with haven/dplyr/fixest packages.")
    medicaid_available = False

Medicaid data loaded successfully
betahat length: 7
sigma dimensions: torch.Size([7, 7])

betahat values:
[-0.005285 -0.011297 -0.002676 -0.001419  0.00034   0.046447  0.069206]


In [34]:
if medicaid_available:
    numPrePeriods_med = 5
    numPostPeriods_med = 2
    
    print("=" * 60)
    print("Medicaid Example: Original CS")
    print("=" * 60)
    
    originalCS_med = hd.constructOriginalCS(
        betahat=betahat_med,
        sigma=sigma_med,
        numPrePeriods=numPrePeriods_med,
        numPostPeriods=numPostPeriods_med
    )
    print(originalCS_med)

Medicaid Example: Original CS
        lb        ub    method Delta
0  0.02851  0.064384  Original  None


In [35]:
if medicaid_available:
    print("=" * 60)
    print("Medicaid Example: DeltaRM Sensitivity")
    print("=" * 60)
    
    delta_rm_med = hd.createSensitivityResults_relativeMagnitudes(
        betahat=betahat_med,
        sigma=sigma_med,
        numPrePeriods=numPrePeriods_med,
        numPostPeriods=numPostPeriods_med,
        Mbarvec=[0.5, 1.0, 1.5, 2.0]
    )
    print(delta_rm_med.to_string(index=False))

Medicaid Example: DeltaRM Sensitivity
       lb       ub method   Delta  Mbar
 0.024002 0.066876   C-LF DeltaRM   0.5
 0.017040 0.072006   C-LF DeltaRM   1.0
 0.008611 0.079335   C-LF DeltaRM   1.5
-0.000550 0.088129   C-LF DeltaRM   2.0


In [36]:
if medicaid_available:
    print("=" * 60)
    print("Medicaid Example: DeltaSD Sensitivity")
    print("=" * 60)
    
    delta_sd_med = hd.createSensitivityResults(
        betahat=betahat_med,
        sigma=sigma_med,
        numPrePeriods=numPrePeriods_med,
        numPostPeriods=numPostPeriods_med,
        Mvec=[0, 0.01, 0.02, 0.03, 0.04, 0.05]
    )
    print(delta_sd_med.to_string(index=False))

Medicaid Example: DeltaSD Sensitivity


/opt/anaconda3/envs/package-testing/lib/python3.8/site-packages/cvxpy/reductions/solvers/solving_chain.py:356: FutureWarning: 
    You specified your problem should be solved by ECOS. Starting in
    CXVPY 1.6.0, ECOS will no longer be installed by default with CVXPY.
    Please either add ECOS as an explicit install dependency to your project
    or switch to our new default solver, Clarabel, by either not specifying a
    solver argument or specifying ``solver=cp.CLARABEL``. To suppress this
    warning while continuing to use ECOS, you can filter this warning using
    Python's ``warnings`` module until you are using 1.6.0.
    
  warnings.warn(ECOS_DEP_DEPRECATION_MSG, FutureWarning)
/opt/anaconda3/envs/package-testing/lib/python3.8/site-packages/cvxpy/problems/problem.py:1407: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


       lb       ub method   Delta    M
 0.025981 0.060688   FLCI DeltaSD 0.00
 0.013161 0.078687   FLCI DeltaSD 0.01
 0.002822 0.090751   FLCI DeltaSD 0.02
-0.007176 0.100749   FLCI DeltaSD 0.03
-0.017176 0.110749   FLCI DeltaSD 0.04
-0.027176 0.120749   FLCI DeltaSD 0.05


In [37]:
if medicaid_available:
    print("=" * 60)
    print("Medicaid Example: Average Effect (l_vec = [0.5, 0.5])")
    print("=" * 60)
    
    l_vec_avg = torch.tensor([[0.5], [0.5]], dtype=torch.float64)
    
    delta_rm_avg = hd.createSensitivityResults_relativeMagnitudes(
        betahat=betahat_med,
        sigma=sigma_med,
        numPrePeriods=numPrePeriods_med,
        numPostPeriods=numPostPeriods_med,
        Mbarvec=[0, 0.5, 1.0, 1.5, 2.0],
        l_vec=l_vec_avg
    )
    print(delta_rm_avg.to_string(index=False))

Medicaid Example: Average Effect (l_vec = [0.5, 0.5])
       lb       ub method   Delta  Mbar
 0.040833 0.074712   C-LF DeltaRM   0.0
 0.032631 0.079704   C-LF DeltaRM   0.5
 0.019792 0.090403   C-LF DeltaRM   1.0
 0.005884 0.103241   C-LF DeltaRM   1.5
-0.008381 0.117150   C-LF DeltaRM   2.0
